# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)  
- Description: Tabular records for 77 cancer survivors with second primary colorectal cancer, including clinical/pathological and molecular characteristics relevant for research and clinical stratification.

In [ ]:
# Ensure that `mlcroissant` library is installed (uncomment line if running in Colab or other environments)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load metadata and print high-level description
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` values.

We'll enumerate all record sets, show their `@id`, and for each, show its fields and field `@id` values used for data extraction.


In [ ]:
def print_recordset_overview(ds):
    print('Available Record Sets:')
    record_sets = ds.record_sets
    for rs in record_sets:
        print(f"- RecordSet: {rs.name} (@id={rs['@id']})")
        print('  Fields:')
        for f in rs.fields:
            data_type = getattr(f, 'data_type', None) or getattr(f, 'type', None) or ''
            print(f"    - {f.name} (@id={f['@id']}), type={data_type}")
        print()

print_recordset_overview(dataset)

## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame` using `mlcroissant`.

We will use the record set and field `@id` values reported above. (If there is only one main tabular record set, we'll focus on extracting that one first.)

In [ ]:
# Find all record sets
record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # mlcroissant efficiently iterates and loads each record
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id} ({len(df)} records)")
    print('-'*40)

# Display columns of the main (first) record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Fields/Columns of RecordSet '@id' {main_record_set_id}:")
    print(list(dataframes[main_record_set_id].columns))
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate:
- Filtering records for a numeric field (e.g. age at diagnosis)
- Normalizing a numeric variable
- Grouping by a categorical variable (e.g. anatomical location) and computing means

**All fields and columns must be referenced by their `@id`.**

> *For demonstration, substitute the correct field @id and column names printed above: for example, if the age-of-diagnosis field is `http://mlcommons.org/croissant/age_at_diagnosis`, use that as the DataFrame column string.*


In [ ]:
df = dataframes[main_record_set_id].copy()

# Example: let's find an age variable (@id or column name may differ; adjust based on printed columns)
# Assume the age field's @id and DataFrame column is 'cr:age_at_second_crc_diagnosis'
# and the anatomical location variable is 'cr:second_crc_anatomical_location'.
numeric_field_id = 'cr:age_at_second_crc_diagnosis'
group_field_id = 'cr:second_crc_anatomical_location'

# Only continue if the field is present
if numeric_field_id in df.columns:
    threshold = 50  # Filter by age > 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} ({len(filtered_df)} records):")
    display(filtered_df.head())

    # Normalize age (z-score)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Field {numeric_field_id} not found in record set {main_record_set_id}.")

## 5. Visualization

Visualize age distributions and the relationship between MSI-H status and anatomical location, using the field `@id` values.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# For inline plotting in Jupyter
%matplotlib inline

# Example fields (substitute correct @id from data overview)
age_col = numeric_field_id  # 'cr:age_at_second_crc_diagnosis'
anatomical_col = group_field_id  # 'cr:second_crc_anatomical_location'
msi_col = 'cr:msi_mmr_status'  # Adjust if a different @id for MSI-H is shown

# 1. Histogram of ages
if age_col in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[age_col].dropna(), bins=12, kde=True)
    plt.xlabel('Age at 2nd CRC Diagnosis')
    plt.title('Distribution of Age at 2nd CRC Diagnosis')
    plt.show()

# 2. Boxplot: Age vs. anatomical location
if all(col in df.columns for col in [age_col, anatomical_col]):
    plt.figure(figsize=(11, 6))
    order = df[anatomical_col].unique()
    # Optional: remove nulls
    sns.boxplot(x=anatomical_col, y=age_col, data=df, order=order)
    plt.xticks(rotation=45)
    plt.ylabel('Age at Diagnosis')
    plt.xlabel('Anatomical Location (@id)')
    plt.title('Age by Anatomical Location of 2nd CRC')
    plt.show()

# 3. Bar plot: MSI-H status by anatomical site
if all(col in df.columns for col in [msi_col, anatomical_col]):
    plt.figure(figsize=(10, 5))
    msi_counts = pd.crosstab(df[anatomical_col], df[msi_col])
    msi_counts.plot(kind='bar', stacked=True)
    plt.title('MSI/MMR Status by Anatomical Location (@id)')
    plt.xlabel('Anatomical Location')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title='MSI/MMR Status')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This notebook demonstrated loading and exploring a clinical dataset using the Croissant `@id` conventions and the `mlcroissant` Python API.
- All record sets, fields, and columns were referenced by their unique `@id` values, supporting robust, schema-aware data workflows.
- You can further extend this workflow for advanced statistical or machine learning analysis as appropriate for clinical research.